# A simple example of a GPT model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.utils.data import DataLoader, Dataset

## Model

In [19]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, n_head, context_size, dropout=0.2):
        super().__init__()
        assert embed_dim % n_head == 0
        self.n_head = n_head
        self.fc_qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.fc_out = nn.Linear(embed_dim, embed_dim)
        # self.mask = torch.triu(torch.ones(context_size, context_size) * torch.tensor(float('-inf')), diagonal=1)
        self.dropout_sc = nn.Dropout(dropout)
        self.dropout_fc = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_size, context_size) * float('-inf'),
                diagonal=1
            )
        )


    def forward(self, x : torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, embed_dim = x.shape
        
        q, k, v = self.fc_qkv(x).split(embed_dim, dim=2)
        
        q = q.view(batch_size, seq_len, self.n_head, embed_dim // self.n_head).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, embed_dim // self.n_head).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, embed_dim // self.n_head).transpose(1, 2)

        raw_scores = q @ k.transpose(2, 3) / math.sqrt(q.shape[-1])

        raw_scores += self.mask[:seq_len, :seq_len]
        
        scores = F.softmax(raw_scores, dim=-1)
        scores = self.dropout_sc(scores)

        attention = scores @ v

        out = attention.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)

        return self.dropout_fc(self.fc_out(out))


class FeedForward(nn.Module):

    def __init__(self, embed_dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, embed_dim * 4)
        self.fc2 = nn.Linear(embed_dim * 4, embed_dim)
        self.dropout_ff = nn.Dropout(dropout)

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return self.dropout_ff(x)
    

class DecoderBlock(nn.Module):

    def __init__(self, embed_dim, n_head, context_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mha = MultiHeadAttention(embed_dim, n_head, context_size, dropout)
        self.ffn = FeedForward(embed_dim, dropout)

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        x = x + self.mha(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
    

class Transformer(nn.Module):

    def __init__(self, embed_dim, n_head, n_dec, context_size, vocab_size, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.p_embed = nn.Embedding(context_size, embed_dim)
        self.decoders = nn.ModuleList(
            [DecoderBlock(embed_dim, n_head, context_size, dropout) for _ in range(n_dec)]
        )
        self.ln = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.shape
        pos = torch.arange(0, seq_len, dtype=torch.int64, device=self.device)
        pos_embed = self.p_embed(pos)
        embed = self.embed(x)
        x = embed + pos_embed
        for block in self.decoders:
            x = block(x)
        x = self.ln(x)
        x = self.fc(x)
        return x

## Downloading the dataset

In [3]:
!wget https://www.gutenberg.org/cache/epub/2701/pg2701.txt

--2026-03-22 14:33:34--  https://www.gutenberg.org/cache/epub/2701/pg2701.txt
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1276266 (1.2M) [text/plain]
Saving to: ‘pg2701.txt’

pg2701.txt          100%[===================>]   1.22M   828KB/s    in 1.5s    

2026-03-22 14:33:37 (828 KB/s) - ‘pg2701.txt’ saved [1276266/1276266]



## Creating the dataset

Since we're using ASCII characters as tokens, we replace non-ASCII characters with spaces.

In [3]:
class TextDataset(Dataset):

    def __init__(self, seq_len):
        super().__init__()
        with open('pg2701.txt', 'r', encoding='utf-8') as f:
            self.data = f.read()
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len - 1
    
    def __getitem__(self, index):
        full = [ord(self.data[i])  if ord(self.data[i]) < 256 else ord(' ') for i in range(index, index + self.seq_len + 1)]
        seq = torch.tensor(full[:-1], dtype=torch.int64)
        target = torch.tensor(full[1:], dtype=torch.int64)
        return seq, target

We will use 75-character sequences for demonstration.

In [77]:
train_dataset = TextDataset(128)
train_loader = DataLoader(train_dataset, 50, shuffle=True)

In [88]:
model = Transformer(
    embed_dim=128,
    n_head=4,
    n_dec=2,
    context_size=256,
    vocab_size=256,
    dropout=0.05
)

In [89]:
summ = 0
for param in model.parameters():
    summ += param.numel()
print(f'Model size: {summ}')

Model size: 495360


In [90]:
model = model.to('cuda')
model = torch.compile(model)

In [91]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=len(train_loader) * 0.25, gamma=0.5)
criterion = nn.CrossEntropyLoss()

There are 50,000 steps in total, but 2,000 steps are sufficient for demonstration.

In [92]:
model.train()

num_epochs = 1

for epoch in range(num_epochs):
    for i, (seq, target) in enumerate(train_loader):
        optimizer.zero_grad()
        seq = seq.to('cuda')
        target = target.to('cuda')
        outputs = model(seq)
        loss = criterion(outputs.view(-1, outputs.size(-1)), target.view(-1))
        loss.backward()
        optimizer.step()
        scheduler.step()
        if (i % 10) == 0:
            print(f'Step: {i + 1}/{len(train_loader)}, Loss: {loss.item():.4f}')
        # if (i == 2000):
        #     break
    print(f'End epoch {epoch+1}, lr = {optimizer.param_groups[0]["lr"]:.6e}')


Step: 1/24764, Loss: 5.7807
Step: 11/24764, Loss: 4.3820
Step: 21/24764, Loss: 3.4449
Step: 31/24764, Loss: 3.1711
Step: 41/24764, Loss: 2.9701
Step: 51/24764, Loss: 2.8819
Step: 61/24764, Loss: 2.7856
Step: 71/24764, Loss: 2.7561
Step: 81/24764, Loss: 2.7057
Step: 91/24764, Loss: 2.6720
Step: 101/24764, Loss: 2.5947
Step: 111/24764, Loss: 2.6154
Step: 121/24764, Loss: 2.5759
Step: 131/24764, Loss: 2.5780
Step: 141/24764, Loss: 2.5789
Step: 151/24764, Loss: 2.5573
Step: 161/24764, Loss: 2.5790
Step: 171/24764, Loss: 2.5342
Step: 181/24764, Loss: 2.5211
Step: 191/24764, Loss: 2.5148
Step: 201/24764, Loss: 2.5533
Step: 211/24764, Loss: 2.5223
Step: 221/24764, Loss: 2.5459
Step: 231/24764, Loss: 2.4989
Step: 241/24764, Loss: 2.5069
Step: 251/24764, Loss: 2.5033
Step: 261/24764, Loss: 2.4882
Step: 271/24764, Loss: 2.4867
Step: 281/24764, Loss: 2.4773
Step: 291/24764, Loss: 2.5070
Step: 301/24764, Loss: 2.4868
Step: 311/24764, Loss: 2.4678
Step: 321/24764, Loss: 2.4873
Step: 331/24764, Loss

KeyboardInterrupt: 

Here we use argmax for deterministic output so we can verify the inference engine's output.

In [95]:
model.eval()

with torch.no_grad():
    s = 'It were '
    print(s, end='')
    for i in range(30):
        s_tok = torch.tensor([[ord(s[i])  if ord(s[i]) < 256 else ord(' ') for i in range(len(s))]], dtype=torch.int64)[:,-50:]
        s_tok = s_tok.to('cuda')
        out = model(s_tok)
        logits = out[:, -1]
        probs = F.softmax(logits, dim=-1)
        # idx_next = torch.multinomial(probs, num_samples=1).view(-1).tolist()
        idx_next = torch.argmax(probs).view(-1).tolist()
        s += chr(idx_next[0])
        print(chr(idx_next[0]), end='')

It were the ship s boat s boats of the

In [53]:
import os

def get_block_weights(i: int,block: DecoderBlock, model_root_path: str) -> None:
    os.mkdir(f'{model_root_path}/blocks/{i}')
    os.mkdir(f'{model_root_path}/blocks/{i}/qkv_proj')
    os.mkdir(f'{model_root_path}/blocks/{i}/out_proj')
    os.mkdir(f'{model_root_path}/blocks/{i}/ln1')
    os.mkdir(f'{model_root_path}/blocks/{i}/ln2')
    os.mkdir(f'{model_root_path}/blocks/{i}/ffn')
    os.mkdir(f'{model_root_path}/blocks/{i}/ffn/in')
    os.mkdir(f'{model_root_path}/blocks/{i}/ffn/out')
    block.mha.fc_qkv.weight.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/qkv_proj/weight.bin')
    block.mha.fc_qkv.bias.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/qkv_proj/bias.bin')
    block.mha.fc_out.weight.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/out_proj/weight.bin')
    block.mha.fc_out.bias.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/out_proj/bias.bin')
    block.mha.mask.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/mask.bin')
    block.ln1.weight.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ln1/weight.bin')
    block.ln1.bias.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ln1/bias.bin')
    block.ln2.weight.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ln2/weight.bin')
    block.ln2.bias.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ln2/bias.bin')
    block.ffn.fc1.weight.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ffn/in/weight.bin')
    block.ffn.fc1.bias.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ffn/in/bias.bin')
    block.ffn.fc2.weight.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ffn/out/weight.bin')
    block.ffn.fc2.bias.detach().numpy().tofile(f'{model_root_path}/blocks/{i}/ffn/out/bias.bin')

def get_model_weghts(model: Transformer, model_root_path: str) -> None:
    os.mkdir(f'{model_root_path}')
    os.mkdir(f'{model_root_path}/blocks')
    os.mkdir(f'{model_root_path}/ln')
    os.mkdir(f'{model_root_path}/fc')
    model.embed.weight.detach().numpy().tofile(f'{model_root_path}/wte.bin')
    model.p_embed.weight.detach().numpy().tofile(f'{model_root_path}/wpe.bin')
    model.ln.weight.detach().numpy().tofile(f'{model_root_path}/ln/weight.bin')
    model.ln.bias.detach().numpy().tofile(f'{model_root_path}/ln/bias.bin')
    model.fc.weight.detach().numpy().tofile(f'{model_root_path}/fc/weight.bin')
    model.fc.bias.detach().numpy().tofile(f'{model_root_path}/fc/bias.bin')
    for i, block in enumerate(model.decoders):
        get_block_weights(i, block, model_root_path)


In [54]:
get_model_weghts(model, './model')

FileExistsError: [Errno 17] File exists: './model'

The executable must be located in the `build` directory.

In [14]:
!./../build/primitiveml --help

Usage: ./../build/primitiveml [PROMPT] [NUM_OF_TOKENS_TO_GENERATE]

Arguments:
  PROMPT                       Text prompt (required).
  NUM_OF_TOKENS_TO_GENERATE    Positive integer (required).

Options:
  -h, --help                   Show this help message and exit.


Now we can test the C implementation. It also uses argmax, so the output should match the previous result.

In [15]:
!./../build/primitiveml "It were" 50

It were the ship and the s stranger of the 
